# PySpark Column Reference Guide
**Author : Mukesh Date : 07-Dec-2025**

## Understanding Different Ways to Reference Columns

In PySpark, there are **three main ways** to reference columns:

1. **String literals**: `"column_name"`
2. **`col()` function**: `col("column_name")`
3. **DataFrame bracket notation**: `df["column_name"]`

This notebook explains **when to use each method** with practical examples.

---

## Setup: Create Spark Session and Sample Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, when, upper, lower, concat
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Create Spark session
spark = SparkSession.builder \
    .appName("Column Reference Guide") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

In [ ]:
# Create sample data
data = [
    ("Alice", 25, "Engineering", 75000),
    ("Bob", 30, "Sales", 65000),
    ("Charlie", 35, "Engineering", 85000),
    ("Diana", 28, "Marketing", 70000),
    ("Eve", 32, "Sales", 68000)
]

df = spark.createDataFrame(data, ["name", "age", "department", "salary"])

print("Sample DataFrame:")
df.show()

---

## Method 1: String Literals `"column_name"`

### ✅ When to Use:
- **Simple column selection** (no transformations)
- **Sorting** (`orderBy`, `sort`)
- **Grouping** (`groupBy`)
- **Dropping columns** (`drop`)
- **Renaming** (as the target name in `withColumnRenamed`)

### ❌ When NOT to Use:
- Column expressions or transformations
- Comparisons or conditions
- Mathematical operations

### Rule of Thumb:
**Use string literals when you're just naming/identifying a column, not transforming it.**

In [ ]:
# ✅ CORRECT: Simple selection
print("1. Select columns (string literals work):")
df.select("name", "age", "salary").show(3)

In [ ]:
# ✅ CORRECT: Sorting
print("2. Sort by column (string literals work):")
df.orderBy("salary", ascending=False).show(3)

In [ ]:
# ✅ CORRECT: Grouping
print("3. Group by column (string literals work):")
df.groupBy("department").count().show()

In [ ]:
# ✅ CORRECT: Dropping columns
print("4. Drop column (string literals work):")
df.drop("age").show(3)

In [ ]:
# ❌ WRONG: Cannot use string literals for expressions
print("5. This will FAIL - string literals don't work for transformations:")
try:
    df.select("name", "salary" * 1.1).show()  # ERROR!
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

---

## Method 2: `col("column_name")` Function

### ✅ When to Use:
- **Column transformations** (applying functions)
- **Expressions** (mathematical operations, comparisons)
- **Conditions** (filtering, when/otherwise)
- **Aliasing** (renaming with `.alias()`)
- **Join conditions**
- **Any operation that returns a Column object**

### Why Use `col()`:
- Returns a **Column object** that supports operations
- Works across different DataFrames
- More flexible than string literals

### Rule of Thumb:
**Use `col()` when you need to perform operations, transformations, or comparisons on columns.**

In [ ]:
# ✅ CORRECT: Column transformations
print("1. Transform column with function:")
df.select(col("name"), upper(col("name")).alias("name_upper")).show(3)

In [ ]:
# ✅ CORRECT: Mathematical operations
print("2. Mathematical operations:")
df.select(
    col("name"),
    col("salary"),
    (col("salary") * 1.1).alias("salary_with_raise")
).show(3)

In [ ]:
# ✅ CORRECT: Filtering with conditions
print("3. Filter with conditions:")
df.filter(col("age") > 30).show()

In [ ]:
# ✅ CORRECT: Complex conditions
print("4. Complex conditions with when/otherwise:")
df.select(
    col("name"),
    col("salary"),
    when(col("salary") > 70000, "High")
    .when(col("salary") > 65000, "Medium")
    .otherwise("Low")
    .alias("salary_category")
).show()

In [ ]:
# ✅ CORRECT: Multiple column operations
print("5. Combine multiple columns:")
df.select(
    concat(col("name"), lit(" - "), col("department")).alias("employee_info")
).show(3)

In [ ]:
# ✅ CORRECT: Join conditions
print("6. Join with conditions:")
df2 = spark.createDataFrame(
    [("Alice", "Manager"), ("Bob", "Associate"), ("Charlie", "Senior")],
    ["emp_name", "title"]
)

df.join(df2, col("name") == col("emp_name"), "inner").show()

---

## Method 3: DataFrame Bracket Notation `df["column_name"]`

### ✅ When to Use:
- **Disambiguating columns** from different DataFrames
- **Self-joins** (referencing same DataFrame twice)
- **When you need to specify which DataFrame** a column belongs to
- **Column names with special characters or spaces**

### Key Difference from `col()`:
- `df["column"]` is **DataFrame-specific** (tied to that DataFrame)
- `col("column")` is **DataFrame-agnostic** (works across DataFrames)

### Rule of Thumb:
**Use `df["column"]` when you need to explicitly specify which DataFrame the column belongs to.**

In [ ]:
# ✅ CORRECT: Disambiguate columns in joins
print("1. Disambiguate columns from different DataFrames:")
df.join(df2, df["name"] == df2["emp_name"], "inner") \
    .select(df["name"], df["department"], df2["title"]) \
    .show()

In [ ]:
# ✅ CORRECT: Self-join (same DataFrame referenced twice)
print("2. Self-join example:")
df_alias1 = df.alias("df1")
df_alias2 = df.alias("df2")

# Join on department, find colleagues in same department
df_alias1.join(
    df_alias2,
    df_alias1["department"] == df_alias2["department"],
    "inner"
).filter(df_alias1["name"] != df_alias2["name"]) \
 .select(
     df_alias1["name"].alias("employee1"),
     df_alias2["name"].alias("employee2"),
     df_alias1["department"].alias("shared_department")  # Alias to avoid ambiguity
 ).show(5)

In [ ]:
# ✅ CORRECT: Column names with spaces or special characters
print("3. Column names with special characters:")
df_special = df.withColumnRenamed("name", "employee name")
df_special.select(df_special["employee name"], df_special["salary"]).show(3)

In [ ]:
# ✅ CORRECT: Operations on DataFrame-specific columns
print("4. Operations with DataFrame bracket notation:")
df.select(
    df["name"],
    (df["salary"] * 0.1).alias("bonus")
).show(3)

---

## Quick Reference Table

| Operation | String `"col"` | `col("col")` | `df["col"]` |
|-----------|----------------|--------------|-------------|
| **Simple select** | ✅ | ✅ | ✅ |
| **Sort/OrderBy** | ✅ | ✅ | ✅ |
| **GroupBy** | ✅ | ✅ | ✅ |
| **Drop column** | ✅ | ❌ | ❌ |
| **Math operations** | ❌ | ✅ | ✅ |
| **Transformations** | ❌ | ✅ | ✅ |
| **Filtering** | ❌ | ✅ | ✅ |
| **Aliasing** | ❌ | ✅ | ✅ |
| **Join conditions** | ❌ | ✅ | ✅ |
| **Disambiguate DFs** | ❌ | ❌ | ✅ |
| **Self-joins** | ❌ | ❌ | ✅ |

---

## Best Practices

### 1. **Default to `col()` for safety**
When in doubt, use `col()`. It works in all scenarios where string literals work, plus many more.

```python
# Good practice
df.select(col("name"), col("age"))
```

### 2. **Use string literals for simple operations**
For basic selection, sorting, and grouping, string literals are cleaner.

```python
# Clean and simple
df.select("name", "age").orderBy("age")
```

### 3. **Use `df["col"]` for disambiguation**
When working with multiple DataFrames (joins, unions), use bracket notation to be explicit.

```python
# Clear which DataFrame each column belongs to
df1.join(df2, df1["id"] == df2["id"])
```

### 4. **Mix methods appropriately**
You can mix methods in the same operation:

```python
df.select(
    "name",                    # String literal
    col("salary") * 1.1,       # col() for operation
    df["department"]           # Bracket for clarity
)
```

---

## Common Mistakes to Avoid

### ❌ Mistake 1: Using strings for expressions

In [ ]:
# ❌ WRONG
try:
    df.filter("age > 30")  # This is SQL syntax, not Python!
except Exception as e:
    print(f"Error: This doesn't work in PySpark DataFrame API")

# ✅ CORRECT
df.filter(col("age") > 30).show(3)

### ❌ Mistake 2: Forgetting `col()` in withColumn

In [ ]:
# ❌ WRONG
try:
    df.withColumn("bonus", "salary" * 0.1)  # String doesn't support *
except Exception as e:
    print(f"Error: {type(e).__name__}")

# ✅ CORRECT
df.withColumn("bonus", col("salary") * 0.1).show(3)

### ❌ Mistake 3: Ambiguous columns in joins

In [ ]:
# Create DataFrames with same column name
df_emp = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df_dept = spark.createDataFrame([(1, "Engineering"), (2, "Sales")], ["id", "dept"])

# ❌ AMBIGUOUS: Which 'id' after join?
joined = df_emp.join(df_dept, "id")
try:
    joined.select("id", "name", "dept").show()  # Works but ambiguous
except Exception as e:
    print(f"Error: {e}")

# ✅ CLEAR: Specify which DataFrame
joined = df_emp.join(df_dept, df_emp["id"] == df_dept["id"])
joined.select(df_emp["id"], df_emp["name"], df_dept["dept"]).show()

---

## Summary

### When to Use Each Method:

#### 🔤 **String Literals** `"column"`
- Simple selection, sorting, grouping
- Dropping columns
- When you're just **naming** a column

#### 🔧 **`col()` Function**
- Transformations and operations
- Filtering and conditions
- Mathematical expressions
- When you're **doing something** with a column

#### 📊 **DataFrame Bracket** `df["column"]`
- Joins with multiple DataFrames
- Self-joins
- When you need to **specify which DataFrame**

### Golden Rule:
**If you're unsure, use `col()` - it's the most versatile and works in almost all scenarios!**

---

## 🎯 Practice Exercise

Try rewriting these operations using different column reference methods:

1. Select name and salary where age > 30
2. Add a new column "tax" that is 20% of salary
3. Join two DataFrames and select specific columns from each

**Happy Sparking! 🚀**